## 02a Experiment 1a: Category Membership - Attributes Task

### 0 Setup

In [10]:
import os
from openai import OpenAI
import random
import pandas as pd
import time
import csv
import ast

### 1 Functions

In [11]:
# Prompt Model to answer Questionnaire

def collect_output(client,prompt):
    response = client.responses.create(
        model="gpt-5.2",
        input= prompt)
    return response


### A. Attributes Task

In [12]:
# Category
category = "Vegetable"

In [13]:
# CATEGORIES & ATTRIBUTES #from rosch1975c
df = pd.read_csv('01_catmemexp_attributestask_items4attributes_2.csv')
items_df = df[df['category'] == f"{category}"]
items_df.head()

,model,category,frequency,items
63,deepseek-chat,Vegetable,most_frequent,"['pumpkin', 'zucchini', 'pea', 'onion', 'eggpl..."
64,deepseek-chat,Vegetable,average_frequency,"['mushroom', 'watercres', 'swiss chard', 'ruta..."
65,deepseek-chat,Vegetable,least_frequent,"['yam', 'shallot', 'spaghetti squash', 'waterc..."
66,gemini-3-pro,Vegetable,most_frequent,"['pumpkin', 'zucchini', 'pea', 'onion', 'eggpl..."
67,gemini-3-pro,Vegetable,average_frequency,"['salsify', 'chayote', 'plantain', 'pinto bean..."


In [ ]:
# Collect Answers from Model

items_reps = 10 #30

# retrieve API 
with open('', 'r') as file:
    #openai.api_key = file.read() 
    client = OpenAI(api_key='')
    #client = OpenAI()   

prompt_outputs = pd.DataFrame(columns=['model', 'prompt', 'prompt_id', 'runtime', 'category', 'frequency','item','output'])
model_output = pd.DataFrame()

for rep in range(items_reps): # max rep == 30

    items_df = items_df[items_df['model'] == f"gpt-5.2"]
    items_list = items_df['items'].tolist()
    items_list = [ast.literal_eval(item) for item in items_list if isinstance(item, str)]
    items_list = [item for sublist in items_list for item in sublist]

    frequency_list_tmp = items_df['frequency'].tolist()
    frequency_list = [freq for freq in frequency_list_tmp for _ in range(6)]
    paired_items = list(zip(items_list, frequency_list))

    random.shuffle(paired_items) 
    shuffled_items_list, shuffled_range_list = zip(*paired_items)
    shuffled_items_list = list(shuffled_items_list)
    shuffled_range_list = list(shuffled_range_list)
        
    for i in range(len(shuffled_items_list)):
        
        current_prompt = f"""
        Write all of the attributes of the object "{shuffled_items_list[i]}" that you can think of. Write the attributes or characteristics you think are characteristic of the object "{shuffled_items_list[i]}".
        Do not add any further details, only write the items that answer the instruction.
        Provide your answer in plain text as a comma-separated list of adjectives.
        """
            
        start_time = time.time() 
        output = collect_output(client, current_prompt)
        end_time = time.time() 
        runtime = end_time - start_time
        model_output = pd.DataFrame({
            'model': 'gpt-5.2',
            'prompt': [current_prompt],
            'prompt_id': [6],
            'runtime': [runtime],
            'category': [category],
            'frequency': [shuffled_range_list[i]],
            'item': [shuffled_items_list[i]],
            'output': [output]})
            
        prompt_outputs = pd.concat([prompt_outputs, model_output], ignore_index=True)

prompt_outputs.head()

C:\Users\AS\AppData\Local\Temp\ipykernel_21028\2897509021.py:53: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  prompt_outputs = pd.concat([prompt_outputs, model_output], ignore_index=True)


,model,prompt,prompt_id,runtime,category,frequency,item,output
0,gpt-5.2,\n Write all of the attributes of the o...,6,3.252948,Vegetable,most_frequent,onion,Response(id='resp_003c7a25bd28425c0069b33c43ac...
1,gpt-5.2,\n Write all of the attributes of the o...,6,1.924232,Vegetable,average_frequency,oregano,Response(id='resp_0726e3a7fda44f200069b33c46cb...
2,gpt-5.2,\n Write all of the attributes of the o...,6,2.492998,Vegetable,least_frequent,sugar snap pea,Response(id='resp_010ad63fdf390ff90069b33c48b5...
3,gpt-5.2,\n Write all of the attributes of the o...,6,2.437502,Vegetable,average_frequency,chive,Response(id='resp_0cacff19356ae3fa0069b33c4b39...
4,gpt-5.2,\n Write all of the attributes of the o...,6,3.377439,Vegetable,average_frequency,black bean,Response(id='resp_00c516f8bafdf2cd0069b33c4da9...


In [ ]:
def extract_text(response):
    if response and response.output:
        return response.output[0].content[0].text
    return None 

prompt_outputs['output'] = prompt_outputs['output'].apply(extract_text)

In [16]:
prompt_outputs

,model,prompt,prompt_id,runtime,category,frequency,item,output
0,gpt-5.2,\n Write all of the attributes of the o...,6,3.252948,Vegetable,most_frequent,onion,"layered, bulbous, round, pungent, aromatic, sh..."
1,gpt-5.2,\n Write all of the attributes of the o...,6,1.924232,Vegetable,average_frequency,oregano,"aromatic, fragrant, pungent, herbaceous, earth..."
2,gpt-5.2,\n Write all of the attributes of the o...,6,2.492998,Vegetable,least_frequent,sugar snap pea,"green, crisp, crunchy, sweet, tender, fresh, e..."
3,gpt-5.2,\n Write all of the attributes of the o...,6,2.437502,Vegetable,average_frequency,chive,"green, slender, thin, long, tubular, hollow, c..."
4,gpt-5.2,\n Write all of the attributes of the o...,6,3.377439,Vegetable,average_frequency,black bean,"black, small, oval, oblong, smooth, glossy, ha..."
...,...,...,...,...,...,...,...,...
175,gpt-5.2,\n Write all of the attributes of the o...,6,2.995686,Vegetable,least_frequent,plantain,"starchy, tropical, banana-like, large, elongat..."
176,gpt-5.2,\n Write all of the attributes of the o...,6,2.715492,Vegetable,most_frequent,pea,"small, round, spherical, green, smooth, firm, ..."
177,gpt-5.2,\n Write all of the attributes of the o...,6,3.407677,Vegetable,average_frequency,nori,"edible, dried, seaweed-based, marine, oceanic,..."
178,gpt-5.2,\n Write all of the attributes of the o...,6,3.627672,Vegetable,most_frequent,eggplant,"purple, violet, deep purple, glossy, shiny, sm..."


In [17]:
# take away NaN Answers
print("Number of rows:", len(prompt_outputs),"\n")
print("Number of rows after NA filtering:", len(prompt_outputs.dropna()))

Number of rows: 180 

Number of rows after NA filtering: 180


In [18]:
#prompt_outputs.to_csv(f'01_catmemexp_attributestask_allmodels_raw_2.csv',index=False, quoting=csv.QUOTE_ALL, encoding='utf-8', mode='a')
prompt_outputs.to_csv(f'gpt_vegetable_10newincomplete.csv',index=False, quoting=csv.QUOTE_ALL, encoding='utf-8', mode='a')